In [ ]:
from dotenv import load_dotenv
import sys
from pathlib import Path

# Get the parent directory of the current working directory
parent_dir = Path.cwd().parent.parent

# Add the parent directory's path to sys.path
# sys.path requires strings, so we convert the Path object
sys.path.append(str(parent_dir))

load_dotenv()

In [ ]:
from config.dir import SODA_HF_REPO
from datasets import load_dataset_builder
from src.dataset.n2d_translate import N2DTranslate

In [ ]:
percentage_to_translate = 0.1
select_columns = ["speakers"]

dataset = {}
ds_builder = load_dataset_builder(SODA_HF_REPO)
for split in ds_builder.info.splits:
    total_num_samples = ds_builder.info.splits[split].num_examples
    num_samples_to_translate = int(total_num_samples * percentage_to_translate)
    print(f"Split: {split}, Total Samples: {total_num_samples}, Samples to Translate: {num_samples_to_translate}")
    dataset[split] = ds_builder.as_dataset(split=split).select(range(num_samples_to_translate))
    dataset[split] = dataset[split].select_columns(select_columns)

In [ ]:
all_names = []
for split in dataset.keys():
    print(f"Processing split: {split}")
    names = set(item for sublist in dataset[split]["speakers"] for item in sublist)
    all_names.extend(names)
all_names = list(set(all_names))

In [ ]:
n2d_translator = N2DTranslate(
    attn_implementation="flash_attention_2",
)

In [ ]:
translated_names = n2d_translator.translate_names(
    all_names,
    batch_size=512,
    save_mapping=True,
    filename="name_mapping.json",
)